# Maia Corpus v9 — Acquisition Engine v0.3

**Real pilot ingestion** from OpenWebMath (mathematics), The Stack Smol (code), and Project Gutenberg (books), with **Google Drive as persistent storage**.

**Safety and provenance:** Dataset availability and licensing can change. This notebook does not treat a dataset-level license as a license for every contained work. Documents without individually verified permissions go to **quarantine**. No FineMath. The v8 corpus is never modified. Token estimates are approximate until MaiaTokenizer-65k is trained.

Run cells in order. First run is deliberately small. Hugging Face streaming requires internet access and sometimes authentication or dataset-specific acceptance. Failures are recorded, not hidden.

In [ ]:
%pip -q install -U "datasets>=3.0" "huggingface_hub>=0.25" requests tqdm

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_4_0/maia-corpus-v9')
assert Path('/content/drive/MyDrive').is_dir(), 'Google Drive not mounted; refusing to write to temporary storage'
for p in ['inputs/math','inputs/code','inputs/books','pilot/accepted','pilot/quarantine','pilot/raw_manifest','pilot/reports']:
    (ROOT/p).mkdir(parents=True, exist_ok=True)
print('Persistent root:', ROOT)

## Configuration

`DRY_RUN=False` enables actual downloads. Keep small limits until you inspect samples. Gutenberg records are **quarantined by default** because public-domain status depends on jurisdiction. The Stack source license metadata is checked against a conservative allowlist, but **human license review remains necessary**. OpenWebMath is quarantined pending document-level rights verification.

In [ ]:
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
import json, hashlib, re, gzip, os, time, traceback
import requests
from tqdm.auto import tqdm

@dataclass
class Config:
    dry_run: bool = False
    math_limit: int = 50
    code_limit: int = 50
    books_limit: int = 10
    max_chars_per_document: int = 150_000
    min_chars: int = 200
    request_timeout: int = 30
    math_dataset: str = 'open-web-math/open-web-math'
    code_dataset: str = 'bigcode/the-stack-smol'
    code_languages: tuple = ('python', 'cpp', 'javascript')
    code_per_language: int = 20
    book_languages: tuple = ('pt', 'en', 'es')
    book_per_language: int = 4
    approved_code_licenses: tuple = ('MIT','Apache-2.0','BSD-2-Clause','BSD-3-Clause','ISC','0BSD','CC0-1.0','Unlicense')
CFG = Config()
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT = ROOT/'pilot'
print(asdict(CFG), RUN_ID)

In [ ]:
def sha256_text(s): return hashlib.sha256(s.encode('utf-8')).hexdigest()
def norm_license(s):
    return re.sub(r'\s+', '', str(s or '')).replace('https://spdx.org/licenses/','').replace('.html','')
def extract_text(record):
    for key in ('text','content','code','body','raw_content'):
        val = record.get(key)
        if isinstance(val, str) and val.strip(): return val
    return None

def row_for(domain, source, external_id, text, license_id=None, url=None, metadata=None):
    text = text[:CFG.max_chars_per_document].replace('\x00','')
    return {'domain':domain,'source':source,'external_id':str(external_id),'url':url,
            'license':license_id,'text':text,'sha256':sha256_text(text),
            'characters':len(text),'approx_tokens_chars_div_4':round(len(text)/4),
            'metadata':metadata or {},'collected_utc':datetime.now(timezone.utc).isoformat()}

accepted, quarantined, errors, stats = [], [], [], []
seen = set()
def add(row, reason=None):
    if row['characters'] < CFG.min_chars: reason='too_short'
    if row['sha256'] in seen: reason='duplicate_in_pilot'
    seen.add(row['sha256'])
    if reason:
        row['quarantine_reason']=reason
        quarantined.append(row)
    else: accepted.append(row)

def safe_iter_hf(dataset, *, name=None, split='train', limit=50):
    from datasets import load_dataset
    kwargs = {'split':split,'streaming':True,'trust_remote_code':False}
    if name is not None: kwargs['name']=name
    ds = load_dataset(dataset, **kwargs)
    for i, rec in enumerate(ds):
        if i >= limit: break
        yield i,rec

def record_error(source, e):
    msg=f'{type(e).__name__}: {e}'
    errors.append({'source':source,'error':msg})
    print('SOURCE ERROR:',source,msg[:500])

## 1. Mathematics — OpenWebMath

Streaming from a public HF dataset identifier. We retain metadata and quarantine all records because a repository-level license does not establish rights to each web page. If the repository changes or requires authorization, the error will be reported.

In [ ]:
if not CFG.dry_run:
    source=CFG.math_dataset
    count=0
    try:
        for idx,rec in safe_iter_hf(source,limit=CFG.math_limit):
            text=extract_text(rec)
            if not text: continue
            url=rec.get('url') or rec.get('source_url')
            lic=rec.get('license') or rec.get('license_id')
            row=row_for('math',source,rec.get('id',idx),text,lic,url,
                        {'original_keys':list(rec.keys())[:30]})
            add(row,'rights_unverified')
            count+=1
    except Exception as e: record_error(source,e)
    stats.append({'source':source,'examined_text_records':count})
print('math:',[x for x in stats if x['source']==CFG.math_dataset])

## 2. Code — The Stack Smol

Stream selected language configurations. Accept only records whose **individual** license identifier is in the conservative allowlist. If license metadata is missing or complex, quarantine. Acceptance here means *pilot metadata filter*, **not final legal clearance**. Check the dataset card and upstream attribution requirements before training.

In [ ]:
if not CFG.dry_run:
    for lang in CFG.code_languages:
        source=f'{CFG.code_dataset}:{lang}'
        count=0
        try:
            for idx,rec in safe_iter_hf(CFG.code_dataset,name=lang,limit=CFG.code_per_language):
                text=extract_text(rec)
                if not text: continue
                lic=rec.get('license') or rec.get('license_id') or rec.get('detected_license')
                raw_lic=str(lic or '')
                lic_clean=norm_license(raw_lic)
                allowed=lic_clean in CFG.approved_code_licenses
                row=row_for('code',CFG.code_dataset,rec.get('hexsha',rec.get('id',f'{lang}-{idx}')),
                            text,raw_lic,rec.get('repo_name') or rec.get('repository'),
                            {'language':lang,'path':rec.get('path'),'original_keys':list(rec.keys())[:30]})
                add(row,None if allowed else 'license_missing_or_not_allowlisted')
                count+=1
        except Exception as e: record_error(source,e)
        stats.append({'source':source,'examined_text_records':count})
print('code:',[x for x in stats if x['source'].startswith(CFG.code_dataset)])

## 3. Books — Gutenberg via Gutendex

Uses the public Gutendex catalog and Gutenberg UTF-8 text downloads. **All books are quarantined pending jurisdiction-specific public-domain review** (Gutenberg US status is not automatically sufficient in Brazil). This notebook downloads only small pilot samples, not entire collections.

In [ ]:
def gutendex_books(language, limit):
    url='https://gutendex.com/books/'
    got=0
    for page in range(3):
        if got>=limit or not url: break
        r=requests.get(url,params={'languages':language} if page==0 else None,
                        timeout=CFG.request_timeout,headers={'User-Agent':'MaiaCorpusResearch/0.3'})
        r.raise_for_status(); data=r.json()
        for b in data.get('results',[]):
            if got>=limit: break
            fmts=b.get('formats',{})
            text_url=next((u for k,u in fmts.items() if 'text/plain' in k and 'utf-8' in k.lower()),None)
            if not text_url: text_url=next((u for k,u in fmts.items() if 'text/plain' in k),None)
            if not text_url: continue
            yield b,text_url
            got+=1
        url=data.get('next')

if not CFG.dry_run:
    for lang in CFG.book_languages:
        source=f'gutenberg:{lang}'; count=0
        try:
            for b,url in gutendex_books(lang,CFG.book_per_language):
                try:
                    r=requests.get(url,timeout=CFG.request_timeout,
                                   headers={'User-Agent':'MaiaCorpusResearch/0.3'})
                    r.raise_for_status()
                    r.encoding=r.apparent_encoding or 'utf-8'
                    txt=r.text
                    row=row_for('books','Project Gutenberg',b['id'],txt,
                                'public_domain_claim_unverified',url,
                                {'language':lang,'title':b.get('title'),
                                 'authors':b.get('authors'), 'copyright':b.get('copyright')})
                    add(row,'jurisdictional_rights_review_required')
                    count+=1
                except Exception as e: record_error(f'{source}:{b.get("id")}',e)
        except Exception as e: record_error(source,e)
        stats.append({'source':source,'examined_text_records':count})
print('books:',[x for x in stats if x['source'].startswith('gutenberg:')])

## 4. Persist results to Google Drive

Writes **JSONL.GZ** records in accepted/quarantine directories. Text is included in the pilot archives. Run manifest records SHA-256, counts, source errors, and configuration. Files are named by RUN_ID, so repeat runs do not overwrite prior results. Writes happen on Drive, not local SSD.

In [ ]:
def save_jsonl_gz(path, rows):
    with gzip.open(path,'wt',encoding='utf-8') as f:
        for row in rows: f.write(json.dumps(row,ensure_ascii=False,default=str)+'\n')

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

saved=[]
for kind,rows in [('accepted',accepted),('quarantine',quarantined)]:
    for domain in ('math','code','books'):
        selected=[r for r in rows if r['domain']==domain]
        path=OUT/kind/f'{domain}_{RUN_ID}.jsonl.gz'
        save_jsonl_gz(path,selected)
        saved.append({'path':str(path.relative_to(ROOT)),'count':len(selected),
                      'bytes':path.stat().st_size,'sha256':sha256_file(path)})

manifest={'project':'Maia Corpus v9','version':'0.3','run_id':RUN_ID,
          'created_utc':datetime.now(timezone.utc).isoformat(),
          'persistent_root':str(ROOT),'config':asdict(CFG),
          'counts':{'accepted':len(accepted),'quarantine':len(quarantined)},
          'source_stats':stats,'errors':errors,'files':saved,
          'limitations':['no automatic legal clearance','exact dedup within this run only',
                         'approximate token counts','no dedup against corpus v8',
                         'streaming HF dataset schemas and availability may change']}
report=OUT/'reports'/f'acquisition_report_{RUN_ID}.json'
report.write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('REPORT:',report)
print('ACCEPTED:',len(accepted),'QUARANTINE:',len(quarantined),'ERRORS:',len(errors))
for s in stats: print(s)
for e in errors: print('ERROR:',e)
print('SAVED FILES:',len(saved))

## 5. Inspect sample documents before scaling

This prints short previews only, and summarizes quarantine reasons. **Do not increase limits** until at least one candidate from each source has been inspected and provenance/legal checks are complete.

In [ ]:
from collections import Counter
print('Accepted by domain:',dict(Counter(x['domain'] for x in accepted)))
print('Quarantine by domain:',dict(Counter(x['domain'] for x in quarantined)))
print('Quarantine reasons:',dict(Counter(x['quarantine_reason'] for x in quarantined)))
for domain in ('math','code','books'):
    items=[x for x in accepted+quarantined if x['domain']==domain]
    if items:
        x=items[0]
        print('\n',domain.upper(),x['source'],'license:',x['license'],'id:',x['external_id'])
        print(x['text'][:550].replace('\n',' '))
    else: print('\n',domain.upper(),': no records — inspect errors above')

### Next steps

1. Inspect source failures (if any); source IDs/configurations may need updates.
2. Review licenses **per document**, attribution requirements, and jurisdiction. Move verified documents from quarantine to accepted through a separately audited promotion step.
3. Add near-duplicate filtering and v8 cross-corpus dedup.
4. Benchmark real MaiaTokenizer-65k fertility after training; then scale in resumable shards with storage quotas and source-level checkpoints.

**This v0.3 is a real acquisition pilot, not a production-scale 2B-token downloader.**